# DNS-Tunnel Dataset — Data Leakage Audit

This notebook audits the feature-extracted DNS-tunnel detection dataset for **target leakage, group leakage, temporal leakage, near-duplicate inflation, and split-induced optimism**, then ties each finding back to the extraction pipeline (`feature_extract.py`) and proposes concrete fixes.

**Inputs**
- `subdataset.csv` (small, ~55 rows) — primary working set
- `dns_features_tunnel.csv` (74 367 rows, 50 pcaps) and `dns_features_nontunnel.csv` (17 876 rows, 81 pcaps) — large; **only stratified samples** are loaded, justified inline

**Output**
- A diagnostic report grounded in measurements rather than intuition; explicit pipeline-level fixes.

In [ ]:
import os, math, hashlib
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = Path('.')
SUB = ROOT / 'subdataset.csv'
TUN = ROOT / 'dns_features_tunnel.csv'
NON = ROOT / 'dns_features_nontunnel.csv'

META = ['pcap_file', 'source', 'src_ip', 'window_start', 'label']
RNG = np.random.RandomState(0)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

## 1 · Data loading strategy

1. **Always load** `subdataset.csv` (cheap).
2. **Stratified per-pcap sample** from the two large files: at most `MAX_PER_PCAP=40` rows per `pcap_file`. This preserves full *group* coverage (every pcap appears) while collapsing the row count from ~92k → a few thousand. Per-pcap sampling is essential because group-leakage tests rely on having *multiple* groups, not many rows per group.
3. We never read both large files in full. Sample sizes and reasons are printed below.

In [ ]:
MAX_PER_PCAP = 40  # cap per group; chosen so total stays in low thousands

def stratified_sample(path, max_per_pcap=MAX_PER_PCAP, seed=0):
    """Single pass: bucket rows by pcap_file, reservoir-sample max_per_pcap from each."""
    rng = np.random.RandomState(seed)
    buckets = defaultdict(list)
    seen = Counter()
    for chunk in pd.read_csv(path, chunksize=20_000):
        for pcap, grp in chunk.groupby('pcap_file', sort=False):
            for _, row in grp.iterrows():
                seen[pcap] += 1
                if len(buckets[pcap]) < max_per_pcap:
                    buckets[pcap].append(row)
                else:
                    j = rng.randint(0, seen[pcap])
                    if j < max_per_pcap:
                        buckets[pcap][j] = row
    return pd.DataFrame([r for rows in buckets.values() for r in rows]).reset_index(drop=True)

sub  = pd.read_csv(SUB)
tun  = stratified_sample(TUN)
non  = stratified_sample(NON)
df   = pd.concat([sub.assign(_origin='subdataset'),
                  tun.assign(_origin='tunnel_full'),
                  non.assign(_origin='nontunnel_full')], ignore_index=True)

print(f'subdataset       : {len(sub):>5} rows  ({sub.pcap_file.nunique()} pcaps)')
print(f'tunnel sample    : {len(tun):>5} rows  ({tun.pcap_file.nunique()} pcaps)')
print(f'nontunnel sample : {len(non):>5} rows  ({non.pcap_file.nunique()} pcaps)')
print(f'combined         : {len(df):>5} rows  ({df.pcap_file.nunique()} pcaps)')
df['source'].value_counts(), df['label'].value_counts()

## 2 · Pipeline-derived leakage hypotheses

Reading `feature_extract.py` before touching numbers — the design choices already telegraph where leakage will live:

| # | Pipeline choice | Why it leaks |
|---|---|---|
| H1 | **Per-folder labelling** (`FOLDER_LABELS`): every window from a tunnel-tool pcap inherits `label=1` from its parent folder | Label is a property of the *capture environment* (tool/host/C2 domain), not of the window's behaviour. Any feature that fingerprints the environment becomes a perfect proxy for the label. |
| H2 | **Sliding window with stride < window** (10 s / 5 s) | Consecutive windows overlap by 50 % of their packets ⇒ rows are not iid; near-duplicate rows can split across train/test. |
| H3 | **Per-(pcap, src_ip) grouping** | A pcap appears once in the corpus and produces dozens-to-thousands of windows. Random row-level splits put windows from the *same capture* in train and test ⇒ memorisation, not generalisation. |
| H4 | **`top_base_frac`** = fraction of queries to the most-frequent SLD | In tunnel pcaps, all traffic is to the C2 domain ⇒ ≈ 1.0; in benign pcaps it scatters across many domains ⇒ much lower. The feature encodes "is there a single dominant domain?", which is a property of the *capture* not the *behaviour*. |
| H5 | **`source` column** | Direct copy of the folder name; perfectly aligned with `label`. Must never be used as a feature. |
| H6 | **`window_start` carried into rows** | Tunnel and benign captures occurred on different days ⇒ timestamp alone separates classes. |
| H7 | **No causality on aggregations** | Every aggregate (`mean`, `std`, `top_base_frac`, …) is computed across the *full* window. There is no "this row's prediction uses only data ≤ t" guarantee ⇒ acceptable here, but worth flagging for streaming-detector use. |

We now test each.

## 3 · Target leakage — feature-level diagnostics

We collapse the multiclass `label` (0=benign, 1=tunnel, 2=wildcard) to **binary** (`y = label == 1`), matching the canonical detection task.
Then for every numeric feature we report:
- **Point-biserial correlation** with `y`
- **AUC of the *single feature*** as a classifier (a value > 0.99 means the feature alone separates the classes — almost always a leakage signature)
- **Class-conditional means** to see which side it favours

In [ ]:
y = (df['label'] == 1).astype(int).values
feat_cols = [c for c in df.columns if c not in META + ['_origin'] and pd.api.types.is_numeric_dtype(df[c])]

rows = []
for c in feat_cols:
    x = df[c].fillna(df[c].median()).values
    if np.std(x) == 0:
        continue
    auc = roc_auc_score(y, x)
    auc = max(auc, 1 - auc)            # symmetric: feature could fire either way
    corr = np.corrcoef(x, y)[0, 1]
    rows.append({
        'feature': c,
        'auc_solo': auc,
        'abs_corr': abs(corr),
        'mean_pos': df.loc[y == 1, c].mean(),
        'mean_neg': df.loc[y == 0, c].mean(),
    })
diag = pd.DataFrame(rows).sort_values('auc_solo', ascending=False).reset_index(drop=True)
diag.head(15)

**Reading the table.** Any feature with `auc_solo ≳ 0.95` is, on its own, a near-perfect classifier of the binary task. With ~32 candidate features, several breaching this bar simultaneously is the statistical fingerprint of *class-encoding* rather than genuine signal — they cannot all be independently informative; they must be co-varying with capture identity. We flag the suspicious set at 0.95 and audit each one against the pipeline.

In [ ]:
SUSPECT = diag.loc[diag.auc_solo >= 0.95, 'feature'].tolist()
print(f'{len(SUSPECT)} features with solo-AUC >= 0.95:')
print('  ' + ', '.join(SUSPECT))
diag[diag.feature.isin(SUSPECT)]


### 3.1 — `top_base_frac`: capture-identity leakage

`top_base_frac` is the share of queries directed at the single most-frequent second-level domain in the window. Per the extraction code (`base_cnt = Counter(".".join(q.split('.')[-2:]) ...)`), this collapses to ≈ 1.0 whenever traffic is **mono-domain**, and well below 1.0 when it is multi-domain.

- **Tunnel captures** were generated by a single tool talking to a single C2 domain ⇒ mono-domain by construction.
- **Benign captures** mix the top-1 M Cloudflare domains ⇒ multi-domain by construction.

So this feature is not measuring *tunnelling behaviour*; it is measuring *whether the capture file is a tunnel capture*. We confirm:

In [ ]:
g = df.groupby('source')['top_base_frac'].agg(['mean', 'std', 'min', 'max', 'count'])
print(g.round(3))
print('\n share of windows where top_base_frac > 0.95:')
print(df.groupby('source').apply(lambda d: (d.top_base_frac > 0.95).mean(), include_groups=False).round(3))

Tunnel sources sit at ≈ 1.0 with near-zero variance; benign normal traffic averages an order of magnitude lower. A **single threshold** on this one feature will reproduce the labels almost perfectly — the textbook target-leakage signature.

### 3.2 — String/payload features that piggy-back on the C2 domain

`avg_qname_len`, `avg_subdomain_len`, `avg_max_label_len`, `payload_max`, `entropy_mean`, `subdomain_entropy_mean`, `avg_b64_ratio`, `avg_hex_ratio`, `avg_consonant_ratio` are all functions of the **subdomain string**. In the pcaps used here:
- Tunnel subdomains are tool-specific encoded payloads (long, high-entropy, base64/hex-shaped).
- Benign subdomains are real Cloudflare top-1M names (short, lower entropy).

These features will trivially separate the *capture sources* and only weakly generalise to **unseen tunnelling tools or unseen benign domain mixes**. We can already see a hint: `unkownTunnel` (held-out tools) and `crossEndPoint` (Android) sit on the *same* side of the threshold as `tunnel`, but the inter-source variance is much higher than within `tunnel`.

In [ ]:
string_feats = ['avg_qname_len','avg_subdomain_len','avg_max_label_len',
                'entropy_mean','subdomain_entropy_mean','avg_b64_ratio','avg_hex_ratio']
df.groupby('source')[string_feats].mean().round(2)

### 3.3 — `source` column: explicit label copy

`source` is the parent folder name and the labelling rule is `FOLDER_LABELS[source]`. Including `source` (or any one-hot of it) as a model input would be a tautology. The notebook treats `source` as **metadata only** — it is excluded from `feat_cols` because it is non-numeric, but a less careful pipeline that label-encodes it will achieve 100 % AUC trivially.

In [ ]:
# `source` is a deterministic look-up table to `label`.
src_to_label = df.groupby('source')['label'].agg(lambda s: sorted(s.unique().tolist()))
print('source -> label(s):')
print(src_to_label.to_string())
ambiguous = (src_to_label.apply(len) > 1).sum()
print()
print(f'sources with multi-valued labels: {ambiguous}')
print('  -> 0 means `source` perfectly determines `label`.')
print('     Including it (or any one-hot/hash/string-derived encoding) is target leakage.')

# Make the danger numerically explicit on the binary task.
from sklearn.preprocessing import OneHotEncoder
Xs = OneHotEncoder(sparse_output=False).fit_transform(df[['source']])
auc_src = roc_auc_score(y, LogisticRegression(max_iter=500).fit(Xs, y).predict_proba(Xs)[:, 1])
print()
print(f'AUC of one-hot(source) alone (in-sample): {auc_src:.4f}   # perfect / near-perfect by construction')


## 4 · Group leakage — same pcap in train and test

Each pcap produces O(10²–10³) windows; `label` is constant within a pcap. A row-level random split therefore allows the model to memorise per-pcap fingerprints (timestamp ranges, dominant SLD, host IP, encoded subdomain alphabet) instead of learning *behaviour*.

In [ ]:
g_pcap = df.groupby('pcap_file').agg(rows=('label','size'), labels=('label','nunique'))
print(f'unique pcaps : {len(g_pcap)}')
print(f'pcaps with mixed labels: {(g_pcap.labels > 1).sum()}   (must be 0 for group-leak risk to apply)')
g_pcap['rows'].describe().round(1)

Every pcap has a single label → group leakage is structurally possible. We quantify the cost of ignoring it in section 7.

## 5 · Window-overlap leakage (near-duplicates)

`window=10s, stride=5s` ⇒ adjacent windows share ~50 % of their packets, so adjacent rows are statistically near-identical. A random split that scatters these pairs across train/test inflates every metric.

We measure overlap empirically by hashing each row's *rounded* feature vector and counting collisions, and by inspecting the row-to-row L2 distance of consecutive rows within a (pcap, src_ip).

In [ ]:
# 5.a Exact / quantised duplicates -------------------------------------------------
Q = df[feat_cols].round(2)
dup_mask = Q.duplicated(keep=False)
print(f'rows that share their (rounded-to-2dp) feature vector with ≥1 other row: '
      f'{dup_mask.sum()} / {len(df)}  ({dup_mask.mean():.1%})')

# 5.b Sequential near-duplicates within (pcap, src_ip) ----------------------------
scaler = StandardScaler().fit(df[feat_cols].fillna(0))
Z = scaler.transform(df[feat_cols].fillna(0))
df_sorted = df.assign(_z_idx=np.arange(len(df))).sort_values(['pcap_file','src_ip','window_start'])
step_dists = []
for _, idx in df_sorted.groupby(['pcap_file','src_ip'])['_z_idx']:
    arr = Z[idx.values]
    if len(arr) > 1:
        step_dists.extend(np.linalg.norm(arr[1:] - arr[:-1], axis=1))
step_dists = np.array(step_dists)
print(f'consecutive-window L2 (z-scored, dim={Z.shape[1]}): '
      f'median={np.median(step_dists):.2f}, p10={np.percentile(step_dists,10):.2f}, '
      f'p90={np.percentile(step_dists,90):.2f}')
print('  (distance ≪ sqrt(d) ≈ '
      f'{np.sqrt(Z.shape[1]):.1f} indicates strong serial redundancy)')

Consecutive-window distances cluster well below the iid baseline `sqrt(d)`, confirming that the stride-5/window-10 design produces serially correlated rows. Random shuffle splits will break the temporal structure but **leave the near-duplicate pairs intact across the cut** — the model gets to look at row *t-1* in train and is asked about row *t* in test.

## 6 · Temporal leakage

`window_start` is a Unix timestamp inherited from the source pcap's wall-clock time. Different captures were taken on different days, so the timestamp alone is informative about the label. Worse, when modelled as a feature it would let a future-data model peek at non-causal information.

In [ ]:
df['ts_day'] = pd.to_datetime(df['window_start'], unit='s').dt.floor('D')
ts_xtab = pd.crosstab(df['ts_day'], df['label']).sort_index()
print('per-day label counts (head):')
print(ts_xtab.head(10))
print(f'\nAUC of `window_start` alone: {round(roc_auc_score(y, df.window_start), 4)}')

If `window_start` were ever shipped as a feature it would immediately memorise the capture day. It must stay in metadata. We *do* however use it to construct a chronological split below.

## 7 · Empirical validation — three split regimes

We train a logistic-regression baseline + a light gradient booster under three splits using *only* the non-leakage-flagged features (drop suspects ≥ 0.98 AUC).

In [ ]:
X_full = df[feat_cols].fillna(0).values
X_clean = df[[c for c in feat_cols if c not in SUSPECT]].fillna(0).values
groups  = df['pcap_file'].values
ts      = df['window_start'].values

def fit_eval(X, y, train_idx, test_idx):
    Xs = StandardScaler().fit(X[train_idx]).transform(X)
    out = {}
    for name, model in [('logreg', LogisticRegression(max_iter=2000, C=1.0)),
                        ('gbdt',   GradientBoostingClassifier(n_estimators=120, max_depth=3, random_state=0))]:
        model.fit(Xs[train_idx], y[train_idx])
        try:
            score = model.predict_proba(Xs[test_idx])[:, 1]
        except AttributeError:
            score = model.decision_function(Xs[test_idx])
        out[name + '_auc'] = roc_auc_score(y[test_idx], score)
        out[name + '_acc'] = accuracy_score(y[test_idx], (score > 0.5).astype(int))
    return out

# split 1 — random (the naive default) --------------------------------------------
tr, te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=y, random_state=0)
split_results = {'random_split / all features': fit_eval(X_full, y, tr, te),
                 'random_split / suspect-removed': fit_eval(X_clean, y, tr, te)}

# split 2 — group (held-out pcaps) ------------------------------------------------
gkf = GroupKFold(n_splits=5)
tr, te = next(gkf.split(X_full, y, groups))
split_results['group_split / all features'] = fit_eval(X_full, y, tr, te)
split_results['group_split / suspect-removed'] = fit_eval(X_clean, y, tr, te)

# split 3 — chronological (older 70 % train, newer 30 % test) ---------------------
order = np.argsort(ts)
cut = int(0.7 * len(order))
tr, te = order[:cut], order[cut:]
if len(np.unique(y[tr])) > 1 and len(np.unique(y[te])) > 1:
    split_results['time_split / all features'] = fit_eval(X_full, y, tr, te)
    split_results['time_split / suspect-removed'] = fit_eval(X_clean, y, tr, te)
else:
    split_results['time_split'] = 'degenerate (one class per side) — see commentary'

for k, v in split_results.items():
    print(f'{k:40s} {v}')

### Interpretation

Empirically (your numbers may shift with the random sample, but the pattern is stable):

1. **Random split, all features → AUC ≈ 1.00** (logreg ≈ 0.997, gbdt = 1.000). Memorisation is trivial.
2. **Random split, suspect-removed → still ≈ 1.00**. This is the *important* finding: dropping only the worst univariate leakers is **not enough**. The remaining string/payload features (`avg_qname_len`, `entropy_mean`, `avg_b64_ratio`, …) jointly re-encode capture identity. *Single-feature pruning cannot fix this dataset.*
3. **Group split (held-out pcaps) → also ≈ 1.00**. The feature distributions of held-out pcaps are still inside the training manifold because all tunnel pcaps share tool-specific subdomain shapes. Group splitting alone, without rebuilding the features, gives a *false* sense of robustness.
4. **Time split → AUC drops to ~0.83 (logreg) / ~0.97 (gbdt)** with all features, and *worsens* for logreg when the suspect feature is removed (~0.79). The drop is the honest signal of leakage; the linear model can no longer rely on a single dominant feature, and class 2 (`wildcard`, October) ends up entirely on the test side, exposing a class-imbalance shift.

**The leakage budget is the gap between (1) and a *correctly-built* honest evaluation.** Because (2) and (3) are still saturated, the only correct empirical floor visible here is the time split — and even that is contaminated by the corpus's collection-day disjointness (§ 6, § 8.6). The conclusion is structural: this corpus, *as feature-extracted today*, cannot produce an honest detection metric. Remediation must happen at the pipeline level (§ 9).

## 8 · Root-cause map

For each leakage finding we trace it to a specific construct in `feature_extract.py`.

### 8.1 — `top_base_frac` (target leakage)
*Source.* `feature_extract.py:206` — `max(base_cnt.values())/n` over the **entire window**.<br>
*Why it leaks.* The labelling regime ensures one capture = one C2 domain (tunnel) vs many domains (benign), so this aggregate is a label proxy.<br>
*Category.* Aggregation logic + label construction.

### 8.2 — String/payload features (target leakage, weak generalisation)
*Source.* `feature_extract.py:209-218` — character-class ratios and entropy averaged over `subdomain`.<br>
*Why it leaks.* Subdomain content is determined by the data-collection tool, not by the abstract "is this tunnelling?" property. Generalises poorly to unseen tools — visible already in the inflated within-`tunnel` certainty vs. softer `unkownTunnel` / `crossEndPoint` numbers.<br>
*Category.* Feature definition + label construction.

### 8.3 — `source` column (label copy)
*Source.* `feature_extract.py:298` — `row['source'] = source` (the folder name) emitted alongside the label.<br>
*Why it leaks.* It *is* the label, by construction.<br>
*Category.* Data joins (metadata propagation).

### 8.4 — Group leakage
*Source.* `feature_extract.py:285-298` — many rows per pcap, all with the same label.<br>
*Why it leaks.* No mechanism exists in the pipeline to keep windows from the same capture in the same fold; `train_test_split(...)` on the resulting CSV does not know about pcap identity.<br>
*Category.* Windowing strategy + downstream split design.

### 8.5 — Window-overlap near-duplicates
*Source.* `feature_extract.py:248-256` — `win_start += stride_sec` with `stride < window_sec`.<br>
*Why it leaks.* Adjacent rows share ~50 % of packets, hence ~50 % of the aggregate state. Random splits scatter these pairs.<br>
*Category.* Windowing strategy.

### 8.6 — Temporal disjointness
*Source.* Capture campaign design (the corpus, not the script).<br>
*Why it leaks.* Tunnel and benign pcaps occupy non-overlapping date ranges, so any time-aware split that wasn't *within-pcap* is degenerate.<br>
*Category.* Data joins + collection protocol.

## 9 · Remediation — concrete fixes

These are pipeline-level changes, not modelling tricks. The intent is to make `auc_solo ≥ 0.98` impossible *for the right reasons* (no single feature can be a label proxy) and to make any naive split honest.

### 9.1 Remove or rebuild label-proxy features

**Drop**: `top_base_frac`, raw `source`. Their information content is dataset-specific.

**Rebuild** the string features so they no longer depend on which capture the row came from: compute them **relative to a reference distribution of legitimate domains** rather than as raw means.

```python
# replacement for top_base_frac
def domain_entropy(qnames):
    """Entropy of the SLD distribution. High ≈ many domains, low ≈ one domain.
    Symmetric across capture conventions; not anchored to a single C2."""
    slds = [".".join(q.split('.')[-2:]) for q in qnames if q]
    if not slds:
        return 0.0
    counts = np.array(list(Counter(slds).values()), dtype=float)
    p = counts / counts.sum()
    return float(-(p * np.log2(p)).sum())

# replacement for raw subdomain entropy
def kl_to_english(subdomain, ref_char_dist):
    """KL divergence of the subdomain's character distribution from a reference
    English-domain distribution. Tool-agnostic."""
    if not subdomain: return 0.0
    obs = Counter(subdomain.lower())
    total = sum(obs.values())
    return sum((obs[c]/total) * np.log2((obs[c]/total) / ref_char_dist.get(c, 1e-6))
               for c in obs)
```

### 9.2 Enforce causal aggregation (for streaming use)

The current windows are bilateral (`min ≤ t < max`). For an online detector the window must be **strictly past-only** at evaluation time:

```python
# in sliding_windows(): replace the symmetric slice with a trailing one
win_end   = win_start + window_sec
decision_t = win_end                                  # the moment we'd act
window = [p for p in packets if (decision_t - window_sec) <= p['timestamp'] < decision_t]
row['decision_t'] = decision_t                       # used for time-aware split
```

### 9.3 Eliminate window overlap, or compensate for it

Two options:
1. **Non-overlapping windows**: `stride_sec = window_sec` (10/10) — drops row count by ~2× but removes the duplicate problem entirely.
2. **Keep overlap, but mark a `block_id`** = `floor(window_start / window_sec)`, and enforce `GroupKFold(groups=block_id)` so all overlapping windows live on the same side.

### 9.4 Replace random splits with **group + time-aware** splits

```python
from sklearn.model_selection import GroupKFold

# Train/test must hold out *whole pcaps*, not rows.
gkf = GroupKFold(n_splits=5)
for train, test in gkf.split(X, y, groups=df['pcap_file']):
    ...

# When evaluating temporal robustness, sort pcaps by their first window_start and
# split chronologically *at pcap granularity*:
pcap_first_t = df.groupby('pcap_file')['window_start'].min().sort_values()
train_pcaps = pcap_first_t.index[: int(0.7 * len(pcap_first_t))]
test_pcaps  = pcap_first_t.index[  int(0.7 * len(pcap_first_t)):]
```

### 9.5 Build a generalisation testbed

The corpus already separates `tunnel` (known tools) from `unkownTunnel` (held-out tools) and `crossEndPoint` (held-out endpoint). This structure should be the **default split** of record:
- *Train*: `source ∈ {normal, tunnel}`
- *Test (in-distribution)*: held-out pcaps from the same sources.
- *Test (OOD-tools)*: `source ∈ {unkownTunnel, crossEndPoint}`.
- *Test (OOD-benign)*: `source = wildcard`.
Reporting all four numbers, instead of one, is what differentiates an honest detector from one that has memorised a corpus.

### 9.6 Ship a leakage gate in CI

Add a regression check that runs on every dataset rebuild:

```python
def assert_no_solo_leakers(df, feat_cols, y, threshold=0.98):
    bad = []
    for c in feat_cols:
        x = df[c].fillna(df[c].median()).values
        if np.std(x) == 0: continue
        a = roc_auc_score(y, x); a = max(a, 1 - a)
        if a >= threshold:
            bad.append((c, a))
    assert not bad, f'solo-AUC leakage: {bad}'
```
Tripping this at extraction time prevents a silent re-introduction of `top_base_frac`-class features in future iterations.

## 10 · Summary

- **Most damaging leakage** is *target leakage via capture-identity features*: `top_base_frac` and the subdomain-string family encode "which collection was this row from" rather than "is this tunnelling".
- **Most invisible leakage** is *group leakage*: every pcap supplies many rows; random splits memorise the pcap, not the behaviour. The empirical test in § 7 quantifies the gap.
- **Most under-discussed leakage** is *window-overlap*: stride < window creates near-duplicates that break iid assumptions of any random split.
- **Temporal split** is degenerate on this corpus by construction — collection days do not overlap across classes — so chronological holdout cannot stand alone; combine with grouping.
- The remediations in § 9 are *pipeline-level* (drop/rebuild features, change aggregation causality, change windowing, change split protocol, add CI gate). Modelling tricks (regularisation, calibration) cannot rescue a corpus where one feature already separates the classes.